In [1]:
!pip install git+https://github.com/huggingface/diffusers
!pip install -U transformers accelerate sentencepiece

  Cloning https://github.com/huggingface/diffusers to /tmp/pip-req-build-ujxxxtaw
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers /tmp/pip-req-build-ujxxxtaw
  Resolved https://github.com/huggingface/diffusers to commit 62b10716093b78028923ad86eb8a8cc787b70aba
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for diffusers: filename=diffusers-0.38.0.dev0-py3-none-any.whl size=5152402 sha256=7b6a1e9e795df9b797b7af5f621429959ef526dbfb2c627b56a5186cce27490a
  Stored in directory: /tmp/pip-ephem-wheel-cache-fne1xrdp/wheels/90/d4/44/a58bc00fb405fefb633b0d9d2307f6e3aec6cc1775d82555d3
Successfully built diffusers
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.37.1
    Uninstalling diffusers-0.37.1:
      Successfully uninstalled diffusers-0.37.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 92.6 MB/s eta

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
project_path = '/content/drive/MyDrive/CASteer_CV'
os.chdir(project_path)

print("Current Working Directory:", os.getcwd())
print("Files in this directory:", os.listdir())

Mounted at /content/drive
Current Working Directory: /content/drive/MyDrive/CASteer_CV
Files in this directory: ['compute_steering_vectors.py', 'generate_casteer.py', 'imagenet_classes.txt', 'construct_prompts.py', 'README.md', '__pycache__', 'controller.py', 'casteer_raw_v1.ipynb', 'cache', 'steering_vectors', 'construct_prompts_mod.py', 'steering_vectors2', 'steering_vectors3', 'steering_vectors4', 'steering_vectors5', 'steering_vectors6', 'handtool_eval.json', 'furniture_eval.json', 'vehicle_eval.json', 'controller_attn_mod.py']


In [2]:
import torch
from diffusers import StableDiffusionPipeline

# Configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "CompVis/stable-diffusion-v1-4"

# Load Pipeline
print("Loading model... this takes about 30-60 seconds...")
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    variant="fp16"
).to(device)

print("Diffusion Model loaded")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading model... this takes about 30-60 seconds...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--CompVis--stable-diffusion-v1-4/snapshots/133a221b8aa7292a167afc5127cb63fb5005638b/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--CompVis--stable-diffusion-v1-4/snapshots/133a221b8aa7292a167afc5127cb63fb5005638b/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Diffusion Model loaded


In [13]:
# @title Attention-Masked Steering + K-Means Subconcept Selection — Cell 4
import os
import pickle
import numpy as np
import torch
import torch.nn.functional as F
from collections import defaultdict
from tqdm.auto import tqdm
from sklearn.cluster import KMeans
from controller import VectorStore, register_vector_control

LOAD_DIR          = '/content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_vecs'
MAIN_CONCEPT_FILE = 'sd14_vehicle.pickle'

SUBCONCEPT_TOKENS = {
    'sd14_airplane.pickle':     ['airplane'],
    'sd14_bicycle.pickle':       ['bicycle'],
    'sd14_boat.pickle':      ['boat'],
    'sd14_bus.pickle':      ['bus'],
    'sd14_car.pickle': ['car'],
    'sd14_motorcycle.pickle':         ['motorcycle'],
    'sd14_scooter.pickle':       ['scooter'],
    'sd14_train.pickle':      ['train'],
    'sd14_truck.pickle':       ['truck'],
    'sd14_van.pickle':         ['van'],
}

MASK_THRESHOLD = 1.3
BETA           = 2
K              = 10    # number of subconcept vectors to keep after K-Means


# ── Token index helper ────────────────────────────────────────────────────────
def get_token_indices(tokenizer, prompt: str, concept_words: list) -> list:
    tokens = tokenizer.tokenize(prompt.lower())
    indices = []
    for i, tok in enumerate(tokens):
        tok_clean = tok.replace('</w>', '')
        for cw in concept_words:
            if cw.lower() in tok_clean:
                indices.append(i + 1)
                break
    return indices


# ── K-Means subconcept selection ──────────────────────────────────────────────
def flatten_sv(sv_dict: dict) -> np.ndarray:
    """
    Flatten a steering vector dict into a single 1-D numpy array.
    Concatenates all per-step, per-place, per-layer vectors.
    Used purely for computing distances in K-Means — not for steering itself.
    """
    parts = []
    for step in sorted(sv_dict.keys()):
        for place in ['down', 'mid', 'up']:
            if place in sv_dict[step]:
                for layer_vec in sv_dict[step][place]:
                    parts.append(layer_vec.flatten())
    return np.concatenate(parts)


def select_subconcepts_kmeans(subconcept_svs: list, k: int, seed: int = 42) -> list:
    """
    Runs K-Means on the flattened steering vectors of the subconcepts.
    For each cluster, selects the subconcept whose vector is closest to
    the cluster centroid as the representative.

    Args:
        subconcept_svs: list of (token_words, sv_dict)
        k:              number of clusters / representatives to keep
        seed:           random seed for reproducibility

    Returns:
        Pruned list of (token_words, sv_dict), length == k.
    """
    if len(subconcept_svs) <= k:
        print(f"  [K-Means] Only {len(subconcept_svs)} subconcepts — skipping selection.")
        return subconcept_svs

    print(f"  [K-Means] Flattening {len(subconcept_svs)} subconcept vectors...")
    flat_vecs = np.stack([flatten_sv(sv_dict) for _, sv_dict in subconcept_svs])
    # shape: [n_subconcepts, total_dims]

    print(f"  [K-Means] Running K-Means with K={k}...")
    kmeans = KMeans(n_clusters=k, random_state=seed, n_init='auto')
    kmeans.fit(flat_vecs)

    # For each cluster, pick the subconcept closest to its centroid
    selected_indices = []
    for cluster_id in range(k):
        members = np.where(kmeans.labels_ == cluster_id)[0]
        centroid = kmeans.cluster_centers_[cluster_id]
        dists = np.linalg.norm(flat_vecs[members] - centroid, axis=1)
        closest = members[np.argmin(dists)]
        selected_indices.append(closest)

    selected = [subconcept_svs[i] for i in sorted(selected_indices)]
    selected_names = [
        '+'.join(subconcept_svs[i][0]) for i in sorted(selected_indices)
    ]
    print(f"  [K-Means] Selected subconcepts: {selected_names}\n")
    return selected


# ── Attention-Masked Vector Store ─────────────────────────────────────────────
class AttentionMaskedVectorStore(VectorStore):
    def __init__(self, main_sv: dict,
                 subconcept_svs: list,
                 tokenizer,
                 beta: float = 2.0,
                 mask_threshold: float = 1.5,
                 device: str = 'cuda'):
        super().__init__(steering_vectors=main_sv, steer=True, device=device)
        self.main_sv        = main_sv
        self.subconcept_svs = subconcept_svs
        self.tokenizer      = tokenizer
        self.beta           = beta
        self.threshold      = mask_threshold
        self._prompt        = ""
        self.attn_weight_cache: dict = {}

    def set_prompt(self, prompt: str):
        self._prompt = prompt

    def _spatial_gate(self, place, layer_idx, token_indices, spatial_size, dtype, device):
        key = (place, layer_idx)
        if key not in self.attn_weight_cache or not token_indices:
            return torch.ones(1, spatial_size, 1, dtype=dtype, device=device)

        attn_map = self.attn_weight_cache[key].to(dtype=dtype, device=device)
        B, H, S, L = attn_map.shape
        cond_map = attn_map[B // 2:]
        avg_map  = cond_map.mean(dim=(0, 1))        # [S, L]

        valid_idx = [i for i in token_indices if i < L]
        if not valid_idx:
            return torch.ones(1, spatial_size, 1, dtype=dtype, device=device)

        concept_attn = avg_map[:, valid_idx].mean(dim=-1)  # [S]
        if S != spatial_size:
            concept_attn = F.interpolate(
                concept_attn.view(1, 1, -1), size=spatial_size,
                mode='linear', align_corners=False
            ).view(-1)

        mean_attn = concept_attn.mean()
        relative_attn = concept_attn / (mean_attn + 1e-6)
        gate = torch.sigmoid((relative_attn - self.threshold) * 5.0)
        return gate.view(1, spatial_size, 1)

    def forward(self, vector: torch.Tensor, place_in_unet: str) -> torch.Tensor:
        if self.steer and place_in_unet in ['up', 'mid', 'down']:
            layer_idx = len(self.step_store[place_in_unet])
            S = vector.size(1)

            # Tier 1: global subtraction for main concept
            num_steer = 0 if len(self.main_sv) == 1 else self.cur_step
            if num_steer in self.main_sv:
                sv_list = self.main_sv[num_steer]
                if place_in_unet in sv_list and layer_idx < len(sv_list[place_in_unet]):
                    sv   = sv_list[place_in_unet][layer_idx]
                    sv_t = torch.tensor(sv, dtype=vector.dtype,
                                        device=self.device).view(1, 1, -1)
                    sim  = torch.clamp(
                        torch.tensordot(vector, sv_t, dims=([2], [2]))
                             .view(vector.size(0), S, 1),
                        min=0.0
                    )
                    vector = vector - self.beta * sim * sv_t.expand(1, S, -1)

            # Tier 2: spatially masked subtraction for K-Means selected subconcepts
            for token_words, sv_dict in self.subconcept_svs:
                num_steer_sub = 0 if len(sv_dict) == 1 else self.cur_step
                if num_steer_sub not in sv_dict:
                    continue
                sv_list = sv_dict[num_steer_sub]
                if place_in_unet not in sv_list:
                    continue
                if layer_idx >= len(sv_list[place_in_unet]):
                    continue

                sv   = sv_list[place_in_unet][layer_idx]
                sv_t = torch.tensor(sv, dtype=vector.dtype,
                                    device=self.device).view(1, 1, -1)
                sim  = torch.clamp(
                    torch.tensordot(vector, sv_t, dims=([2], [2]))
                         .view(vector.size(0), S, 1),
                    min=0.0
                )
                token_indices = get_token_indices(
                    self.tokenizer, self._prompt, token_words
                )
                gate = self._spatial_gate(
                    place_in_unet, layer_idx, token_indices,
                    S, vector.dtype, self.device
                )
                vector = vector - self.beta * gate * sim * sv_t.expand(1, S, -1)

        self.step_store[place_in_unet].append(
            vector.data.cpu().numpy()[len(vector) // 2:].mean(axis=0).mean(axis=0)
        )
        return vector


# ── Load vectors ──────────────────────────────────────────────────────────────
def _stem(fname):
    return os.path.splitext(fname)[0].lower().replace('sd14_', '').replace('_', ' ')

def load_all_steering_vectors_from_dir(load_dir, main_concept_file):
    all_files = [f for f in os.listdir(load_dir) if f.endswith('.pickle')]
    if main_concept_file not in all_files:
        raise FileNotFoundError(f"'{main_concept_file}' not found in {load_dir}")
    other_files = sorted(f for f in all_files if f != main_concept_file)
    result = {}
    bar = tqdm([main_concept_file] + other_files, desc="Loading steering vectors", unit="file")
    for fname in bar:
        bar.set_postfix_str(fname)
        with open(os.path.join(load_dir, fname), 'rb') as fh:
            result[fname] = pickle.load(fh)
        tqdm.write(f"Loaded '{fname}'")
    return result


print("Loading steering vectors...")
sv_registry = load_all_steering_vectors_from_dir(LOAD_DIR, MAIN_CONCEPT_FILE)
print(f"Loaded {len(sv_registry)} vectors.\n")

main_sv = sv_registry[MAIN_CONCEPT_FILE]

# Build full subconcept list
all_subconcept_svs = []
for fname, sv_dict in sv_registry.items():
    if fname == MAIN_CONCEPT_FILE:
        continue
    stem   = _stem(fname)
    tokens = next((v for k, v in SUBCONCEPT_TOKENS.items() if k in stem), [stem])
    all_subconcept_svs.append((tokens, sv_dict))

print(f"Total subconcepts loaded: {len(all_subconcept_svs)}")

# K-Means selection: reduce to K representative subconcepts
print(f"Selecting K={K} representative subconcepts via K-Means...")
subconcept_svs = select_subconcepts_kmeans(all_subconcept_svs, k=K)
print(f"Subconcepts after selection: {len(subconcept_svs)}\n")


# ── Generation helpers ────────────────────────────────────────────────────────
def generate_multi_concept_erased(pipe, prompt, num_denoising_steps,
                                   all_steering_vectors=None,
                                   beta=BETA, device='cuda'):
    controller = AttentionMaskedVectorStore(
        main_sv        = main_sv,
        subconcept_svs = subconcept_svs,
        tokenizer      = pipe.tokenizer,
        beta           = beta,
        mask_threshold = MASK_THRESHOLD,
        device         = device,
    )
    controller.set_prompt(prompt)
    register_vector_control(pipe.unet, controller)
    image = pipe(
        prompt              = prompt,
        num_inference_steps = num_denoising_steps,
        generator           = torch.Generator(device=device),
    ).images[0]
    return image


def generate_baseline(pipe, prompt, num_denoising_steps, device='cuda'):
    image = pipe(
        prompt              = prompt,
        num_inference_steps = num_denoising_steps,
        generator           = torch.Generator(device=device),
    ).images[0]
    return image

print("Cell 4 ready — Attention-Masked Steering with K-Means subconcept selection.")
all_sv = list(sv_registry.values())  # alias for API compatibility with eval cells

Loading steering vectors...


Loading steering vectors:   0%|          | 0/11 [00:00<?, ?file/s]

Loaded 'sd14_vehicle.pickle'
Loaded 'sd14_airplane.pickle'
Loaded 'sd14_bicycle.pickle'
Loaded 'sd14_boat.pickle'
Loaded 'sd14_bus.pickle'
Loaded 'sd14_car.pickle'
Loaded 'sd14_motorcycle.pickle'
Loaded 'sd14_scooter.pickle'
Loaded 'sd14_train.pickle'
Loaded 'sd14_truck.pickle'
Loaded 'sd14_van.pickle'
Loaded 11 vectors.

Total subconcepts loaded: 10
Selecting K=10 representative subconcepts via K-Means...
  [K-Means] Only 10 subconcepts — skipping selection.
Subconcepts after selection: 10

Cell 4 ready — Attention-Masked Steering with K-Means subconcept selection.


In [14]:
# @title Evaluation Pipeline — CLIP Score per Category (with image saving)
import json
import os
import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

!pip install git+https://github.com/openai/CLIP.git

import clip

# ── Config ────────────────────────────────────────────────────────────────────
EVAL_JSON_PATH = '/content/drive/MyDrive/CASteer_CV/vehicle_eval.json'
IMAGES_BASE_DIR = '/content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_kmattn2_images'
BETA           = 2
NUM_STEPS      = 50

# Categories: robustness vs. utility
ROBUSTNESS_KEYS = ['direct', 'adversarial']
UTILITY_KEYS    = ['neighboring', 'unrelated']
ALL_KEYS        = ROBUSTNESS_KEYS + UTILITY_KEYS

# ── Create output folders ─────────────────────────────────────────────────────
for key in ALL_KEYS:
    os.makedirs(os.path.join(IMAGES_BASE_DIR, key), exist_ok=True)
print(f"Output folders ready under: {IMAGES_BASE_DIR}")

# ── Load CLIP model ───────────────────────────────────────────────────────────
print("Loading CLIP model...")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()
print("CLIP loaded.\n")



  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-cpuoa4mr
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-cpuoa4mr
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
Output folders ready under: /content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_kmattn2_images
Loading CLIP model...
CLIP loaded.



In [15]:

def compute_clip_score(image: Image.Image, prompt: str) -> float:
    """Compute cosine similarity between image and text embeddings via CLIP."""
    img_tensor  = clip_preprocess(image).unsqueeze(0).to(device)
    text_tokens = clip.tokenize([prompt], truncate=True).to(device)

    with torch.no_grad():
        img_feat  = clip_model.encode_image(img_tensor)
        txt_feat  = clip_model.encode_text(text_tokens)
        img_feat  = img_feat / img_feat.norm(dim=-1, keepdim=True)
        txt_feat  = txt_feat / txt_feat.norm(dim=-1, keepdim=True)
        score     = (img_feat * txt_feat).sum(dim=-1).item()
    return score


# ── Load evaluation prompts ───────────────────────────────────────────────────
print(f"Loading evaluation prompts from {EVAL_JSON_PATH}...")
with open(EVAL_JSON_PATH, 'r') as f:
    eval_data = json.load(f)

for key in ALL_KEYS:
    assert key in eval_data, f"Key '{key}' not found in eval JSON."
    print(f"  {key}: {len(eval_data[key])} prompts")
print()

# ── Run evaluation ────────────────────────────────────────────────────────────
category_scores = {key: [] for key in ALL_KEYS}

for category in ALL_KEYS:
    prompts = eval_data[category]
    out_dir = os.path.join(IMAGES_BASE_DIR, category)

    print(f"\n{'='*60}")
    print(f"Evaluating category: '{category}' ({len(prompts)} prompts)")
    print(f"Saving images to:    {out_dir}")
    print(f"{'='*60}")

    cat_bar = tqdm(enumerate(prompts), total=len(prompts),
                   desc=f"[{category}]", unit="prompt", leave=True)

    for i, prompt in cat_bar:
        cat_bar.set_postfix_str(f'"{prompt[:40]}…"')

        # Generate steered image
        image = generate_multi_concept_erased(
            pipe, prompt, NUM_STEPS,
            all_sv, beta=BETA, device=device
        )

        # Save image — filename is zero-padded index + truncated prompt slug
        slug = prompt[:50].strip().replace(' ', '_').replace('/', '-')
        img_filename = f"{i:03d}_{slug}.png"
        image.save(os.path.join(out_dir, img_filename))

        # Compute CLIP score
        score = compute_clip_score(image, prompt)
        category_scores[category].append(score)

        tqdm.write(f"  [{i+1:>3}/{len(prompts)}] CLIP={score:.4f}  |  {prompt[:60]}")


# ── Aggregate results ─────────────────────────────────────────────────────────
avg_scores = {key: np.mean(vals) for key, vals in category_scores.items()}

robustness_avg = np.mean([avg_scores[k] for k in ROBUSTNESS_KEYS])
utility_avg    = np.mean([avg_scores[k] for k in UTILITY_KEYS])

print("done")



Loading evaluation prompts from /content/drive/MyDrive/CASteer_CV/vehicle_eval.json...
  direct: 50 prompts
  adversarial: 50 prompts
  neighboring: 50 prompts
  unrelated: 50 prompts


Evaluating category: 'direct' (50 prompts)
Saving images to:    /content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_kmattn2_images/direct


[direct]:   0%|          | 0/50 [00:00<?, ?prompt/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  [  1/50] CLIP=0.3643  |  A photorealistic red sports car speeding through a neon-lit 


  0%|          | 0/50 [00:00<?, ?it/s]

...

  [ 48/50] CLIP=0.2795  |  A plate of pasta with rich sauce


  0%|          | 0/50 [00:00<?, ?it/s]

  [ 49/50] CLIP=0.3044  |  A snowy cabin in the woods


  0%|          | 0/50 [00:00<?, ?it/s]

  [ 50/50] CLIP=0.3555  |  A phoenix rising from flames in fantasy art
done


In [ ]:

# ── Print results table ───────────────────────────────────────────────────────
print("\n\n" + "="*65)
print("EVALUATION RESULTS — Average CLIP Score per Category")
print("="*65)
print(f"{'Category':<20} {'Purpose':<15} {'Avg CLIP Score':>15}  {'#Prompts':>9}")
print("-"*65)
for key in ROBUSTNESS_KEYS:
    print(f"  {key:<18} {'Robustness':<15} {avg_scores[key]:>15.4f}  {len(category_scores[key]):>9}")
print(f"  {'--- Robustness ---':<18} {'Overall':<15} {robustness_avg:>15.4f}")
print()
for key in UTILITY_KEYS:
    print(f"  {key:<18} {'Utility':<15} {avg_scores[key]:>15.4f}  {len(category_scores[key]):>9}")
print(f"  {'--- Utility ---':<18} {'Overall':<15} {utility_avg:>15.4f}")
print("="*65)

# ── Save raw scores to disk ───────────────────────────────────────────────────
results_out = {
    'avg_scores':        avg_scores,
    'robustness_avg':    robustness_avg,
    'utility_avg':       utility_avg,
    'per_prompt_scores': {k: list(map(float, v)) for k, v in category_scores.items()}
}
out_path = '/content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_attn2_results.json'
with open(out_path, 'w') as f:
    json.dump(results_out, f, indent=2)
print(f"\nFull results saved to: {out_path}")
print(f"Generated images saved under: {IMAGES_BASE_DIR}")



EVALUATION RESULTS — Average CLIP Score per Category
Category             Purpose          Avg CLIP Score   #Prompts
-----------------------------------------------------------------
  direct             Robustness               0.2789         50
  adversarial        Robustness               0.2815         50
  --- Robustness --- Overall                  0.2802

  neighboring        Utility                  0.3062         50
  unrelated          Utility                  0.3152         50
  --- Utility ---    Overall                  0.3107

Full results saved to: /content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_kmattn2_results.json
Generated images saved under: /content/drive/MyDrive/CASteer_CV/steering_vectors6/vehicle_kmattn2_images


In [17]:
from google.colab import runtime
runtime.unassign()